# Day 97: Capstone: FastAPI + vLLM Inference Server

**Layer:** Capstone


## Concept Overview

Build a production-grade FastAPI wrapper around vLLM with health checks, Prometheus metrics, request tracing, and graceful shutdown. This is the deployable artifact for the capstone.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Production FastAPI Server

Implement the full server with all production requirements.


In [ ]:
print("Production FastAPI + vLLM server structure:")
server_code = """
import time, asyncio
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse
from prometheus_client import Counter, Histogram, Gauge, generate_latest
from pydantic import BaseModel
from typing import Optional, List
import uvicorn

app = FastAPI(title="Inference Server", version="1.0")

# Prometheus metrics
REQUEST_COUNT = Counter("inference_requests_total", "Total requests", ["status"])
TTFT_HIST = Histogram("inference_ttft_seconds", "TTFT", buckets=[.05,.1,.2,.5,1.,2.,5.])
TPOT_HIST = Histogram("inference_tpot_ms", "TPOT ms", buckets=[5,10,20,50,100,200])
QUEUE_GAUGE = Gauge("inference_queue_depth", "Queue depth")

class ChatRequest(BaseModel):
    model: str
    messages: List[dict]
    max_tokens: int = 256
    temperature: float = 0.7
    stream: bool = False

@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": True}

@app.get("/metrics")
async def metrics():
    return generate_latest()

@app.post("/v1/chat/completions")
async def chat(req: ChatRequest, request: Request):
    t0 = time.time()
    QUEUE_GAUGE.inc()
    try:
        # vllm_engine.generate() call would go here
        # Simulated response
        ttft = time.time() - t0
        TTFT_HIST.observe(ttft)
        REQUEST_COUNT.labels(status="success").inc()
        return {"choices": [{"message": {"role": "assistant", "content": "..."}}]}
    except Exception as e:
        REQUEST_COUNT.labels(status="error").inc()
        raise HTTPException(500, str(e))
    finally:
        QUEUE_GAUGE.dec()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000, workers=1)
"""
print(server_code)


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Build a production-grade FastAPI wrapper around vLLM with health checks, Prometheus metrics, request tracing, and graceful shutdown.
- Build a production-grade FastAPI wrapper around vLLM with health checks, Prometh....
- Day 97 implementation complete.
